# Product search and similar products

A catalog of 20,000 Amazon products, 150 real customer queries, and for each one human relevance judgments: E (*Exact*, the product asked for), S (*Substitute*, an acceptable substitute), C (*Complement*, an accessory, such as a case for a phone) or I (*Irrelevant*).

1. Write the evaluation measures
2. A baseline: BM25
3. Dense search, exact then approximate (HNSW)
4. Hybrid search
5. Reranking with a cross-encoder
6. Compare, and read the failures
7. Similar products

This notebook needs a GPU: *Runtime > Change runtime type > T4 GPU*.

## Data and models

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s esci-products-subset
%pip install -q bm25s faiss-cpu

In [ ]:
import time

import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder, SentenceTransformer

products = pd.read_parquet("esci-products-subset/products.parquet")
queries = pd.read_parquet("esci-products-subset/queries.parquet")
qrels = pd.read_parquet("esci-products-subset/qrels.parquet")
products["text"] = products.title + ". " + products.brand + ". " + products.bullets
print(len(products), "products,", len(queries), "queries,", len(qrels), "judgments")

embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device="cuda").half()
embedder.max_seq_length = 128  # the start of a product listing is enough, and encoding is twice as fast
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda", max_length=512)
reranker.model.half()
print("Models loaded")

## 1. Writing the evaluation measures

For each query, a search engine returns an ordered list of product IDs. We measure it with:

- **recall at 100**: the share of the query's *Exact* products found in the top 100 results (what a reranking step will see)
- **nDCG at 10**: the quality of the top 10 results (what the customer sees). Each result brings a gain according to its judgment (E: 1, S: 0.1, C: 0.01, I or unjudged: 0), divided by $\log_2(\text{rank} + 1)$; the sum (DCG) is divided by that of the best possible ranking (ideal DCG)

Complete `recall_at` and `ndcg_at`, then run the test cell. `gains[query_id]` maps each product judged for a query to its gain.

In [ ]:
GAIN = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}
gains = {
    query_id: dict(zip(group.product_id, group.esci_label.map(GAIN)))
    for query_id, group in qrels.groupby("query_id")
}


def recall_at(ranking: list[str], query_gains: dict[str, float], k: int = 100) -> float:
    ...


def ndcg_at(ranking: list[str], query_gains: dict[str, float], k: int = 10) -> float:
    ...

### Solution

In [ ]:
def recall_at(ranking: list[str], query_gains: dict[str, float], k: int = 100) -> float:
    exact = {product for product, gain in query_gains.items() if gain == 1.0}
    return len(exact & set(ranking[:k])) / len(exact)


def ndcg_at(ranking: list[str], query_gains: dict[str, float], k: int = 10) -> float:
    dcg = sum(query_gains.get(product, 0.0) / np.log2(rank + 2) for rank, product in enumerate(ranking[:k]))
    ideal = sorted(query_gains.values(), reverse=True)[:k]
    idcg = sum(gain / np.log2(rank + 2) for rank, gain in enumerate(ideal))
    return dcg / idcg

In [ ]:
toy = {"a": 1.0, "b": 0.1, "c": 0.0, "d": 1.0}
assert recall_at(["a", "x", "d"], toy) == 1.0
assert recall_at(["a", "x", "y"], toy, k=2) == 0.5
assert abs(ndcg_at(["a", "d", "b"], toy) - 1.0) < 1e-9
assert abs(ndcg_at(["x", "a", "d"], toy) - 0.6728001281246765) < 1e-9


def evaluate(run: dict[int, list[str]]) -> pd.Series:
    """The mean recall@100 and nDCG@10 of a run: query ID -> ranked product IDs."""
    return pd.Series({
        "recall@100": np.mean([recall_at(run[q], gains[q]) for q in run]),
        "nDCG@10": np.mean([ndcg_at(run[q], gains[q]) for q in run]),
    })


print("Measures correct")

## 2. A baseline: BM25

[`bm25s`](https://github.com/xhluca/bm25s) indexes a corpus for BM25 in seconds. Index the products' texts (`bm25s.tokenize(texts, stopwords="en")`, then `bm25s.BM25().index(...)`), retrieve each query's top 100 products (`retriever.retrieve(bm25s.tokenize(queries, stopwords="en"), k=100)`, which returns product indices and their scores), store them in `runs["BM25"]` as `{query_id: [product_id, ...]}`, and evaluate.

In [ ]:
runs = {}

In [ ]:
# Your code here

### Solution

In [ ]:
import bm25s

retriever = bm25s.BM25()
retriever.index(bm25s.tokenize(products.text.tolist(), stopwords="en"))
indices, _ = retriever.retrieve(bm25s.tokenize(queries["query"].tolist(), stopwords="en"), k=100)
runs["BM25"] = {
    query_id: products.product_id.iloc[row].tolist()
    for query_id, row in zip(queries.query_id, indices)
}
evaluate(runs["BM25"])

## 3. Dense search

Encode the products (`embedder.encode_document`, `batch_size=32`) and the queries (`embedder.encode_query`). The vectors are normalized: the cosine is a dot product. Find each query's 100 closest products with a matrix product and `np.argsort`, store the result in `runs["dense"]` and evaluate.

In [ ]:
# Your code here

### Solution

In [ ]:
start = time.time()
product_vectors = embedder.encode_document(products.text.tolist(), batch_size=32, show_progress_bar=True).astype(np.float32)
query_vectors = embedder.encode_query(queries["query"].tolist()).astype(np.float32)
print(f"{time.time() - start:.0f} s to encode the catalog")

scores = query_vectors @ product_vectors.T
top = np.argsort(-scores, axis=1)[:, :100]
runs["dense"] = {
    query_id: products.product_id.iloc[row].tolist()
    for query_id, row in zip(queries.query_id, top)
}
evaluate(runs["dense"])

Comparing a query with every product is exact, but too expensive with millions of products. An **HNSW** index (a neighborhood graph on several levels) finds the nearest neighbors **approximately**, much faster.

Build a `faiss.IndexHNSWFlat(dimension, 32, faiss.METRIC_INNER_PRODUCT)` index, add the products' vectors to it, and search the queries' 100 nearest neighbors (`index.search(query_vectors, 100)`). Measure the search time against exact search, and the share of the 100 exact neighbors HNSW finds. Try `index.hnsw.efSearch = 16`, then `128`: what does this parameter control?

In [ ]:
# Your code here

### Solution

In [ ]:
import faiss

index = faiss.IndexHNSWFlat(product_vectors.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
index.add(product_vectors)
for ef in (16, 64, 128):
    index.hnsw.efSearch = ef
    start = time.time()
    _, approximate = index.search(query_vectors, 100)
    elapsed = time.time() - start
    overlap = np.mean([len(set(a) & set(b)) / 100 for a, b in zip(approximate, top)])
    print(f"efSearch {ef:3}: {elapsed * 1000:.0f} ms, {overlap:.1%} of the exact neighbors")

start = time.time()
np.argsort(-(query_vectors @ product_vectors.T), axis=1)[:, :100]
print(f"exact search: {(time.time() - start) * 1000:.0f} ms")

# efSearch is the size of the candidate list explored in the graph: larger is more accurate
# and slower. With 20,000 products, exact search is still fast; HNSW becomes essential with
# millions of vectors.

## 4. Hybrid search

Reciprocal Rank Fusion (RRF) combines several rankings without comparing their scores: each product gets $\sum 1 / (60 + \text{rank})$ over the rankings it appears in (ranks from 1). Write `rrf(*rankings)`, fuse BM25 and dense search for each query into `runs["hybrid"]` (keep the top 100), and evaluate.

In [ ]:
# Your code here

### Solution

In [ ]:
from collections import defaultdict


def rrf(*rankings: list[str], k: int = 60) -> list[str]:
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, product in enumerate(ranking, start=1):
            scores[product] += 1 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)


runs["hybrid"] = {
    query_id: rrf(runs["BM25"][query_id], runs["dense"][query_id])[:100]
    for query_id in queries.query_id
}
evaluate(runs["hybrid"])

# Fusion finds more exact products in its top 100 (recall@100): that's what the next step,
# reranking, uses. Its nDCG@10 isn't necessarily better than dense search's when BM25 is
# clearly weaker.

## 5. Reranking with a cross-encoder

A cross-encoder reads the query and the product **together**: more accurate, but one model pass per pair. For the first 100 queries, rerank the top 50 hybrid results with `reranker.predict([(query, text), ...], batch_size=32)`, then append the next 50 in hybrid order. Store the result in `runs["hybrid + reranking"]` and compare with hybrid on the **same** 100 queries.

In [ ]:
# Your code here

### Solution

In [ ]:
text_of = dict(zip(products.product_id, products.text))
query_of = dict(zip(queries.query_id, queries["query"]))
subset = queries.query_id.iloc[:100]

start = time.time()
reranked = {}
for query_id in subset:
    candidates = runs["hybrid"][query_id][:50]
    scores = reranker.predict([(query_of[query_id], text_of[p]) for p in candidates], batch_size=32)
    order = [candidates[i] for i in np.argsort(-scores)]
    reranked[query_id] = order + runs["hybrid"][query_id][50:]
runs["hybrid + reranking"] = reranked
print(f"{time.time() - start:.0f} s to rerank 100 queries")

pd.DataFrame({
    "hybrid": evaluate({q: runs["hybrid"][q] for q in subset}),
    "hybrid + reranking": evaluate(reranked),
})

## 6. Comparing, and reading the failures

Gather every method's measures in a table, on the 100 reranked queries. Then find the queries where BM25 clearly beats dense search (in nDCG@10), and the reverse: read them. What do they have in common?

In [ ]:
# Your code here

### Solution

In [ ]:
print(pd.DataFrame({name: evaluate({q: run[q] for q in subset}) for name, run in runs.items()}).round(3))

per_query = pd.DataFrame({
    "query": [query_of[q] for q in queries.query_id],
    "BM25": [ndcg_at(runs["BM25"][q], gains[q]) for q in queries.query_id],
    "dense": [ndcg_at(runs["dense"][q], gains[q]) for q in queries.query_id],
})
per_query["gap"] = per_query.dense - per_query.BM25
print("BM25 wins:", per_query.nsmallest(8, "gap")["query"].tolist())
print("Dense wins:", per_query.nlargest(8, "gap")["query"].tolist())

# BM25 often wins on references, brands and exact names; dense search on queries worded
# differently from the listings, or with typos.

## 7. Similar products

On a product's page, we suggest similar products: its nearest neighbors in the embedding space, with no query. Write `similar(product_id, k=5, same_brand=False)`, which prints the titles of a product's `k` neighbors (without the product itself), optionally restricted to the same brand. Try it on a few products.

And since the model is multilingual: search the English catalog with queries in another language, such as French (« casque de moto avec écouteurs », « gamelle pour chien surélevée »).

In [ ]:
# Your code here

### Solution

In [ ]:
position = {product: i for i, product in enumerate(products.product_id)}


def similar(product_id: str, k: int = 5, same_brand: bool = False) -> None:
    i = position[product_id]
    scores = product_vectors @ product_vectors[i]
    scores[i] = -np.inf
    if same_brand:
        scores[products.brand.to_numpy() != products.brand[i]] = -np.inf
    print("Product:", products.title[i], "| brand:", products.brand[i])
    for j in np.argsort(-scores)[:k]:
        if np.isfinite(scores[j]):
            print(f"  {scores[j]:.3f} {products.title[j][:100]}")


# Products whose brand has other products in the catalog.
brands = products.brand[products.brand != ""].value_counts()
candidates = products[products.brand.isin(brands[brands >= 5].index)]
for product_id in candidates.product_id.sample(3, random_state=0):
    similar(product_id)
    similar(product_id, same_brand=True)

for french in ("casque de moto avec écouteurs", "gamelle pour chien surélevée"):
    scores = product_vectors @ embedder.encode_query(french).astype(np.float32)
    print(french, "->", products.title.iloc[np.argsort(-scores)[:3]].str[:80].tolist())